# Monday: word embeddings and sentence embeddings

An embedding is a row of numbers for a word or a text. You try word embeddings and sentence embeddings on a few sentences, then use both to search the AITA replies.

Work with a partner. Fill in the cells that start with a comment and use the check comments to confirm your result. Switch who types after Part 2. Make a copy named `week09_live_workspace_mine.ipynb`.

In [ ]:
# Install sentence-transformers, gensim, and the other optional NLP packages into the course environment.
# Run this once on your own computer. It can take a few minutes. Skip it on Colab, which already has them.
!uv sync --frozen --group optional-nlp

In [ ]:
import re
import pandas as pd
import gensim.downloader
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

pd.set_option("display.max_colwidth", 300)  # show more of each reply

## 1. Word embeddings

In [ ]:
# GloVe: word embeddings trained on Wikipedia and news, with 50 numbers per word.
# The first run downloads about 66 MB.
glove = gensim.downloader.load("glove-wiki-gigaword-50")

In [ ]:
glove["wife"]  # the 50 numbers for "wife"

In [ ]:
glove.most_similar("wife")  # the words with the closest rows. Check: daughter 0.96, husband 0.95

In [ ]:
# Find the closest words to "housework". Check: chores first, 0.79

Are the closest words to *wife* synonyms? What do they have in common?

## 2. From words to sentences

Run these cells and read the scores. Part 3 uses the same code on the AITA replies.

In [ ]:
# One row for a sentence: the average of the rows for its words.
# Stop words and words GloVe does not know are left out.
def average_word_vector(text):
    words = re.findall(r"[a-z]+", text.lower())
    words = [w for w in words if w in glove and w not in ENGLISH_STOP_WORDS]
    return sum(glove[w] for w in words) / len(words)

In [ ]:
query = "You are wrong."
examples = pd.DataFrame({"text": [
    "You are not wrong.",
    "You did nothing wrong.",
    "You are mistaken.",
    "The cat sat on the mat.",
]})

In [ ]:
example_word_rows = [average_word_vector(text) for text in examples["text"]]  # one row per sentence
examples["word_score"] = cosine_similarity([average_word_vector(query)], example_word_rows)[0]
examples.round(2)  # Check: 1.0, 0.95, 0.71, 0.43

In [ ]:
# A sentence model reads the whole sentence and gives one row of 384 numbers for it.
# The first run downloads about 90 MB.
model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
example_sentence_rows = model.encode(examples["text"].tolist())  # model.encode needs a list
examples["sentence_score"] = cosine_similarity(model.encode([query]), example_sentence_rows)[0]
examples.round(2)  # Check: 0.83, 0.39, 0.69, 0.01

Which method scores "You are mistaken." above "You did nothing wrong."? Why does the word average give "You are not wrong." a score of 1?

## 3. Search the AITA replies

Use the code from Part 2 on the replies.

In [ ]:
url = "https://raw.githubusercontent.com/macss-berkeley/compss-211a/main/docs/interactives/data/week08/aita_replies.csv"
comments = pd.read_csv(url)
comments["marker"] = comments["body"].str.upper().str.extract(r"^(NTA|YTA)\b")
replies = comments.dropna(subset=["marker"]).reset_index(drop=True)  # keep the replies that start with NTA or YTA
replies["marker"].value_counts()  # Check: NTA 657, YTA 378

In [ ]:
# Make reply_word_rows: one average word vector for each reply in replies["body"]. Check: len(reply_word_rows) is 1035

In [ ]:
# Make reply_sentence_rows with model.encode. This can take a minute. Check: reply_sentence_rows.shape is (1035, 384)

In [ ]:
query = "You should break up with him."

In [ ]:
# Score the query against reply_word_rows and save the scores in replies["word_score"]. Check: the highest score is 0.86

In [ ]:
replies.sort_values("word_score", ascending=False).head(5)[["idstr", "word_score", "body"]]  # Check: the first is t1_einm6sy

In [ ]:
# Score the query against reply_sentence_rows and save the scores in replies["sentence_score"]. Check: the highest score is 0.75

In [ ]:
# Show the five replies with the highest sentence_score. Check: the first is t1_ej7bo1p

Which list is about breaking up? Which words of the query are not stop words?

## 4. Your own search

Write a query about another topic in the replies, such as housework or a wedding. Show the five closest replies with each method. Which method finds more replies about your topic?

In [ ]:
# Your query and the five closest replies with each method

## If you have time: plot the replies

PCA turns each row of 384 numbers into two numbers, so each reply can be drawn as a point.

In [ ]:
import seaborn as sns
from sklearn.decomposition import PCA

points = PCA(n_components=2).fit_transform(reply_sentence_rows)  # two numbers per reply
replies["x"] = points[:, 0]
replies["y"] = points[:, 1]
sns.scatterplot(data=replies, x="x", y="y", hue="marker", s=10)  # Check: most YTA replies are on one side

Is that because of what the YTA replies say, or because they start with YTA? Remove the marker, encode again, and plot.

In [ ]:
no_marker = replies["body"].str.replace(r"^(NTA|YTA)\b", "", regex=True, case=False)  # each reply without its NTA or YTA
# Encode no_marker, run PCA on the result, and plot it the same way. Check: the two groups overlap

Save the notebook, restart the kernel, and run every cell.